
# Project 1 ART Data Exploration

HSLU NLP HS26 Abductive Commonsense Reasoning

## Objectives
- Explore the ART dataset
- Understand the input format and labels
- Analyze text lengths and vocabulary
- Prepare preprocessing decisions for MLP and GRU models


In [2]:

from datasets import load_dataset

# Load the HSLU ART dataset
dataset = load_dataset("amarfurt/art-nlp-hs26")

# Show dataset structure
print(dataset)

# Inspect column names
print("\nColumns:", dataset["train"].column_names)

# Inspect one training example
print("\nFirst example:")
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 16801
    })
    validation: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 1532
    })
})

Columns: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label']

First example:
{'observation_1': 'A baby panda was lost in the woods.', 'observation_2': 'He was reunited with his parents, who also lived at the zoo.', 'hypothesis_1': 'He was never lost.', 'hypothesis_2': 'All the people from the zoo looked for him.', 'label': 2}


In [3]:

import pandas as pd

# Check label distribution for each split
for split in ["train", "validation", "test"]:
    labels = pd.Series(dataset[split]["label"])

    print(f"\n{split.upper()}")
    print(labels.value_counts().sort_index())
    print("\nPercentages:")
    print(labels.value_counts(normalize=True).sort_index() * 100)



TRAIN
1    8461
2    8340
Name: count, dtype: int64

Percentages:
1    50.360098
2    49.639902
Name: proportion, dtype: float64

VALIDATION
1    508
2    492
Name: count, dtype: int64

Percentages:
1    50.8
2    49.2
Name: proportion, dtype: float64

TEST
1    781
2    751
Name: count, dtype: int64

Percentages:
1    50.979112
2    49.020888
Name: proportion, dtype: float64


In [4]:

import re
import pandas as pd

fields = [
    "observation_1",
    "observation_2",
    "hypothesis_1",
    "hypothesis_2"
]

# Simple word-level tokenizer (preliminary)
def tokenize(text):
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text.lower())

# Calculate token lengths using training data only
lengths = []

for example in dataset["train"]:
    segment_lengths = [
        len(tokenize(example[field]))
        for field in fields
    ]

    # Add 3 separator tokens
    total_length = sum(segment_lengths) + 3
    lengths.append(segment_lengths + [total_length])

length_df = pd.DataFrame(
    lengths,
    columns=["O1", "O2", "H1", "H2", "Total"]
)

# Show statistics
print(length_df.describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).round(2))


             O1        O2        H1        H2     Total
count  16801.00  16801.00  16801.00  16801.00  16801.00
mean       9.14     10.48      9.48      9.50     41.61
std        2.60      2.86      3.13      3.16      7.52
min        3.00      3.00      1.00      1.00     20.00
50%        9.00     11.00      9.00      9.00     41.00
90%       13.00     14.00     13.00     13.00     51.00
95%       14.00     15.00     15.00     15.00     55.00
99%       15.00     16.00     20.00     20.00     64.00
max       19.00     20.00     35.00     34.00     92.00


In [5]:

from collections import Counter

# Build vocabulary using training data only
word_counts = Counter()

for example in dataset["train"]:
    for field in fields:
        tokens = tokenize(example[field])
        word_counts.update(tokens)

# Vocabulary statistics
total_tokens = sum(word_counts.values())
vocab_size = len(word_counts)

rare_words = sum(
    1 for count in word_counts.values()
    if count == 1
)

print("Total tokens:", total_tokens)
print("Vocabulary size:", vocab_size)
print("Words occurring only once:", rare_words)
print("Percentage of vocabulary occurring once:",
      round(rare_words / vocab_size * 100, 2), "%")

print("\n20 most common tokens:")
print(word_counts.most_common(20))


Total tokens: 648679
Vocabulary size: 16752
Words occurring only once: 5197
Percentage of vocabulary occurring once: 31.02 %

20 most common tokens:
[('.', 65608), ('the', 27649), ('to', 23083), ('a', 20893), ('was', 15793), ('and', 13201), ('her', 9464), ('he', 8280), ('his', 8122), ('i', 7536), ('she', 6778), ('in', 6710), (',', 6077), ('for', 6046), ('it', 5826), ('of', 5700), ('had', 5531), ('on', 4765), ('at', 3795), ('my', 3694)]


In [6]:

SEP = "[SEP]"

def prepare_example(example):
    tokens = []

    for i, field in enumerate(fields):
        if i > 0:
            tokens.append(SEP)

        tokens.extend(tokenize(example[field]))

    label = example["label"] - 1

    return {
        "tokens": tokens,
        "label": label
    }


# Test with the first training example
example = prepare_example(dataset["train"][0])

print("Input:")
print(" ".join(example["tokens"]))

print("\nSequence length:", len(example["tokens"]))
print("Label:", example["label"])
print("Correct hypothesis:", example["label"] + 1)


Input:
a baby panda was lost in the woods . [SEP] he was reunited with his parents , who also lived at the zoo . [SEP] he was never lost . [SEP] all the people from the zoo looked for him .

Sequence length: 41
Label: 1
Correct hypothesis: 2


In [7]:

import gensim.downloader as api

model_name = "fasttext-wiki-news-subwords-300"

# Read metadata (does not download the model)
info = api.info(model_name)

print("Model:", model_name)
print("Dimensions:", info["parameters"]["dimension"])
print("Vocabulary size:", info["num_records"])
print("Download size:", round(info["file_size"] / 1024**3, 2), "GiB")
print("Training corpus:", info["base_dataset"])


Model: fasttext-wiki-news-subwords-300
Dimensions: 300
Vocabulary size: 999999
Download size: 0.94 GiB
Training corpus: Wikipedia 2017, UMBC webbase corpus and statmt.org news dataset (16B tokens)


In [8]:

import gensim.downloader as api

# Download and load pretrained fastText embeddings
fasttext = api.load("fasttext-wiki-news-subwords-300")

print("Model loaded successfully!")
print("Vocabulary size:", len(fasttext))
print("Embedding dimensions:", fasttext.vector_size)

# Inspect a word embedding
word = "panda"

print(f"\nEmbedding for '{word}' (first 10 values):")
print(fasttext[word][:10])


[=-------------------------------------------------] 3.0% 29.1/958.4MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[====================------------------------------] 41.7% 399.3/958.4MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=========================-------------------------] 50.9% 488.0/958.4MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



... (15 outputs left out)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[===========================================-------] 87.4% 837.8/958.4MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 958.5/958.4MB downloaded
Model loaded successfully!
Vocabulary size: 999999
Embedding dimensions: 300

Embedding for 'panda' (first 10 values):
[ 0.082003 -0.015127 -0.053782 -0.079441 -0.087394 -0.033851  0.045585
 -0.15176  -0.06694   0.064976]


In [9]:

# Check which ART tokens have pretrained fastText vectors
known_words = [
    word for word in word_counts
    if word in fasttext.key_to_index
]

unknown_words = [
    word for word in word_counts
    if word not in fasttext.key_to_index
]

# Vocabulary coverage
vocab_coverage = len(known_words) / len(word_counts) * 100

# Token coverage (weighted by frequency)
known_tokens = sum(word_counts[word] for word in known_words)
token_coverage = known_tokens / total_tokens * 100

print(f"Vocabulary size: {len(word_counts):,}")
print(f"Known words: {len(known_words):,}")
print(f"Unknown words: {len(unknown_words):,}")
print(f"Vocabulary coverage: {vocab_coverage:.2f}%")

print(f"\nToken coverage: {token_coverage:.2f}%")

# Most frequent unknown tokens
print("\n20 most frequent unknown tokens:")
print(
    sorted(
        [(word, word_counts[word]) for word in unknown_words],
        key=lambda x: x[1],
        reverse=True
    )[:20]
)


Vocabulary size: 16,752
Known words: 15,007
Unknown words: 1,745
Vocabulary coverage: 89.58%

Token coverage: 98.40%

20 most frequent unknown tokens:
[("didn't", 1604), ("couldn't", 602), ("wasn't", 437), ("gina's", 194), ("wouldn't", 130), ("year's", 100), ("doesn't", 92), ("hadn't", 88), ("amy's", 85), ("friend's", 85), ("bob's", 68), ("i'm", 65), ("he'd", 61), ("don't", 60), ("it's", 58), ("can't", 56), ("tim's", 56), ("tom's", 54), ("she'd", 52), ("mary's", 50)]


In [10]:

import re
from collections import Counter

def tokenize_v2(text):
    text = text.lower().replace("’", "'")

    # Handle irregular negative contractions
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bshan't\b", "shall not", text)

    # Handle regular negative contractions
    # didn't -> did not, wasn't -> was not, etc.
    text = re.sub(r"\b([a-z]+)n't\b", r"\1 not", text)

    # Tokenize words and punctuation
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text)


# Test our improved tokenizer
examples = [
    "John didn't go to school.",
    "She couldn't find her keys.",
    "He wasn't hungry.",
    "I can't believe it."
]

for sentence in examples:
    print(sentence)
    print("->", tokenize_v2(sentence))
    print()


John didn't go to school.
-> ['john', 'did', 'not', 'go', 'to', 'school', '.']

She couldn't find her keys.
-> ['she', 'could', 'not', 'find', 'her', 'keys', '.']

He wasn't hungry.
-> ['he', 'was', 'not', 'hungry', '.']

I can't believe it.
-> ['i', 'can', 'not', 'believe', 'it', '.']



In [11]:

word_counts_v2 = Counter()

for example in dataset["train"]:
    for field in fields:
        word_counts_v2.update(
            tokenize_v2(example[field])
        )

known_v2 = [
    word for word in word_counts_v2
    if word in fasttext.key_to_index
]

total_v2 = sum(word_counts_v2.values())

vocab_coverage_v2 = (
    len(known_v2) / len(word_counts_v2) * 100
)

token_coverage_v2 = (
    sum(word_counts_v2[w] for w in known_v2)
    / total_v2 * 100
)

print(f"Old vocabulary coverage: {vocab_coverage:.2f}%")
print(f"New vocabulary coverage: {vocab_coverage_v2:.2f}%")

print(f"\nOld token coverage: {token_coverage:.2f}%")
print(f"New token coverage: {token_coverage_v2:.2f}%")

# Examine remaining unknown words
unknown_v2 = [
    word for word in word_counts_v2
    if word not in fasttext.key_to_index
]

print("\nMost frequent remaining unknown tokens:")
print(sorted(
    [(w, word_counts_v2[w]) for w in unknown_v2],
    key=lambda x: x[1],
    reverse=True
)[:20])


Old vocabulary coverage: 89.58%
New vocabulary coverage: 89.68%

Old token coverage: 98.40%
New token coverage: 98.90%

Most frequent remaining unknown tokens:
[("gina's", 194), ("year's", 100), ("amy's", 85), ("friend's", 85), ("bob's", 68), ("i'm", 65), ("he'd", 61), ("it's", 58), ("tim's", 56), ("tom's", 55), ("she'd", 52), ("mary's", 50), ("jim's", 50), ("mother's", 48), ("sam's", 47), ("i've", 46), ("neighbor's", 45), ("mom's", 44), ("lucy's", 43), ("dan's", 42)]


In [12]:

# Special tokens
PAD = "[PAD]"
UNK = "[UNK]"
SEP = "[SEP]"

# Initialize vocabulary
vocab = {
    PAD: 0,
    UNK: 1,
    SEP: 2
}

# Add training vocabulary covered by fastText
for word, count in word_counts_v2.most_common():
    if word in fasttext.key_to_index:
        vocab[word] = len(vocab)

print("Final vocabulary size:", len(vocab))

print("\nSpecial tokens:")
for token in [PAD, UNK, SEP]:
    print(f"{token}: {vocab[token]}")

print("\nExample word IDs:")
for word in ["panda", "lost", "woods", "reunited"]:
    print(f"{word}: {vocab.get(word, vocab[UNK])}")


Final vocabulary size: 15009

Special tokens:
[PAD]: 0
[UNK]: 1
[SEP]: 2

Example word IDs:
panda: 3359
lost: 146
woods: 733
reunited: 4991


In [13]:

def encode_example(example):
    tokens = []

    for i, field in enumerate(fields):
        if i > 0:
            tokens.append(SEP)

        tokens.extend(tokenize_v2(example[field]))

    # Convert tokens to vocabulary IDs
    input_ids = [
        vocab.get(token, vocab[UNK])
        for token in tokens
    ]

    # Convert label from {1, 2} to {0, 1}
    label = example["label"] - 1

    return {
        "input_ids": input_ids,
        "label": label
    }


encoded = encode_example(dataset["train"][0])

print("Token IDs:")
print(encoded["input_ids"])

print("\nSequence length:", len(encoded["input_ids"]))
print("Label:", encoded["label"])
print("Number of unknown tokens:",
      encoded["input_ids"].count(vocab[UNK]))


Token IDs:
[6, 229, 3359, 7, 146, 14, 4, 733, 3, 2, 10, 7, 4991, 24, 11, 137, 15, 187, 515, 455, 22, 4, 599, 3, 2, 10, 7, 65, 146, 3, 2, 39, 4, 186, 46, 4, 599, 164, 16, 37, 3]

Sequence length: 41
Label: 1
Number of unknown tokens: 0


In [14]:

import numpy as np
import torch
import torch.nn as nn

embedding_dim = fasttext.vector_size

# Initialize embedding matrix
embedding_matrix = np.zeros(
    (len(vocab), embedding_dim),
    dtype=np.float32
)

# Copy pretrained fastText vectors
for word, idx in vocab.items():
    if word in [PAD, UNK, SEP]:
        continue

    embedding_matrix[idx] = fasttext[word]

# [UNK]: mean of the available word embeddings
embedding_matrix[vocab[UNK]] = embedding_matrix[3:].mean(axis=0)

# [SEP]: fixed random vector for separating text segments
rng = np.random.default_rng(42)
embedding_matrix[vocab[SEP]] = rng.normal(
    0, 0.05, embedding_dim
)

# [PAD] remains a zero vector

# Create frozen PyTorch embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    torch.from_numpy(embedding_matrix),
    freeze=True,
    padding_idx=vocab[PAD]
)

print("Embedding matrix shape:", embedding_layer.weight.shape)
print("Trainable:", embedding_layer.weight.requires_grad)
print("PAD vector is zero:",
      torch.all(embedding_layer.weight[vocab[PAD]] == 0).item())


Embedding matrix shape: torch.Size([15009, 300])
Trainable: False
PAD vector is zero: True


In [15]:

# Convert our example IDs into a PyTorch tensor
input_ids = torch.tensor(
    encoded["input_ids"],
    dtype=torch.long
)

# Look up the pretrained embeddings
vectors = embedding_layer(input_ids)

print("Input IDs shape:", input_ids.shape)
print("Embedding output shape:", vectors.shape)


Input IDs shape: torch.Size([41])
Embedding output shape: torch.Size([41, 300])


In [16]:

from torch.utils.data import Dataset, DataLoader

class ARTDataset(Dataset):
    def __init__(self, hf_dataset):
        # Encode all examples once
        self.examples = [
            encode_example(example)
            for example in hf_dataset
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# Create PyTorch datasets
train_dataset = ARTDataset(dataset["train"])
val_dataset = ARTDataset(dataset["validation"])

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

print("\nFirst encoded example:")
print(train_dataset[0])


Training examples: 16801
Validation examples: 1000

First encoded example:
{'input_ids': [6, 229, 3359, 7, 146, 14, 4, 733, 3, 2, 10, 7, 4991, 24, 11, 137, 15, 187, 515, 455, 22, 4, 599, 3, 2, 10, 7, 65, 146, 3, 2, 39, 4, 186, 46, 4, 599, 164, 16, 37, 3], 'label': 1}


In [17]:

from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    # Convert token IDs into tensors
    sequences = [
        torch.tensor(item["input_ids"], dtype=torch.long)
        for item in batch
    ]

    # Store original sequence lengths
    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    # Dynamically pad to longest sequence in batch
    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=vocab[PAD]
    )

    # Convert labels into tensor
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    return {
        "input_ids": padded_sequences,
        "lengths": lengths,
        "labels": labels
    }


In [18]:

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

# Inspect one training batch
batch = next(iter(train_loader))

print("Input IDs shape:", batch["input_ids"].shape)
print("Lengths shape:", batch["lengths"].shape)
print("Labels shape:", batch["labels"].shape)

print("\nFirst 5 sequence lengths:")
print(batch["lengths"][:5])

print("\nLabels:")
print(batch["labels"][:10])

print("\nPadding tokens in batch:",
      (batch["input_ids"] == vocab[PAD]).sum().item())


Input IDs shape: torch.Size([32, 62])
Lengths shape: torch.Size([32])
Labels shape: torch.Size([32])

First 5 sequence lengths:
tensor([42, 41, 56, 35, 48])

Labels:
tensor([1, 0, 0, 0, 1, 1, 1, 1, 1, 0])

Padding tokens in batch: 603
